# MIA interaction overview

Reusable counts and work-mode shares for the main MIA interaction parquet.

Definitions used throughout:

- **Attempt**: one parquet row.
- **Unique student / exercise / classroom**: a distinct non-null ID.
- **Target modes**: `zpdes` and `playlist` (configurable below).
- **Attempt share between target modes**: attempts in a mode divided by all `zpdes` + `playlist` attempts in the same scope.
- **Student share between target modes**: distinct students seen in a mode divided by students seen in either target mode in the same scope. Students may use both modes, so these two percentages need not sum to 100%. The exclusive participation table resolves that overlap.

Playlist rows have no raw module title in this dataset. Their module is recovered from `config_mia.json` through `exercise_id`; unresolved rows remain visible as `Unmapped` rather than being dropped.

In [1]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

# Set this explicitly when more than one parquet is present. Otherwise the newest is used.
PARQUET_PATH: Path | None = None
TARGET_MODES = ("zpdes", "playlist")
EXPORT_RESULTS = False

cwd = Path.cwd().resolve()
if (cwd / "data_mia").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data_mia").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Run from the project root or notebooks/ directory.")

DATA_DIR = PROJECT_ROOT / "data_mia"
CONFIG_PATH = DATA_DIR / "config_mia.json"
if PARQUET_PATH is None:
    candidates = sorted(DATA_DIR.glob("*.parquet"))
    if not candidates:
        raise FileNotFoundError(f"No parquet found in {DATA_DIR}")
    PARQUET_PATH = max(candidates, key=lambda path: path.stat().st_mtime)
    if len(candidates) > 1:
        print(f"Found {len(candidates)} parquets; using newest: {PARQUET_PATH.name}")

PARQUET_PATH = Path(PARQUET_PATH).resolve()
print(f"Parquet: {PARQUET_PATH}")
print(f"Config:  {CONFIG_PATH}")

Parquet: C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_MIA\986-neurips-mia_20260415_100024.parquet
Config:  C:\Users\ocler\Documents\Académique\Inria\GAIMHE\Code\visu2\data_mia\config_mia.json


## Build the exercise-to-module lookup

The config contains canonical module, objective, activity, and activity-to-exercise metadata. Exercises assigned to multiple activities are acceptable; exercises assigned to multiple modules are treated as ambiguous and excluded from automatic module assignment.

In [2]:
def title_value(payload: dict, key: str, fallback: str) -> str:
    title = payload.get("title") or {}
    value = title.get(key) if isinstance(title, dict) else None
    return str(value).strip() if value else fallback


with CONFIG_PATH.open(encoding="utf-8-sig") as file:
    config = json.load(file)["config"]

module_by_code: dict[str, dict[str, str]] = {}
module_by_id: dict[str, dict[str, str]] = {}
for payload in config["module"].values():
    module_id = str(payload.get("id") or "").strip()
    module_code = str(payload.get("code") or "").strip()
    if not module_id or not module_code:
        continue
    record = {
        "module_id": module_id,
        "module_code": module_code,
        "module_title": title_value(payload, "short", module_code),
    }
    module_by_code[module_code] = record
    module_by_id[module_id] = record

exercise_modules: defaultdict[str, set[str]] = defaultdict(set)
for payload in config["activity"].values():
    activity_code = str(payload.get("code") or "").strip()
    match = re.match(r"^(M\d+)O", activity_code)
    module = module_by_code.get(match.group(1)) if match else None
    if module is None:
        continue
    for exercise_id in payload.get("learning_items") or []:
        exercise_id = str(exercise_id).strip()
        if exercise_id:
            exercise_modules[exercise_id].add(module["module_id"])

ambiguous_exercises = {
    exercise_id: module_ids
    for exercise_id, module_ids in exercise_modules.items()
    if len(module_ids) > 1
}
exercise_rows = []
for exercise_id, module_ids in exercise_modules.items():
    if len(module_ids) != 1:
        continue
    module = module_by_id[next(iter(module_ids))]
    exercise_rows.append({"exercise_id": exercise_id, **module})

module_lookup = pd.DataFrame(module_by_id.values()).drop_duplicates("module_id")
exercise_lookup = pd.DataFrame(exercise_rows)
print(f"Config modules: {len(module_lookup):,}")
print(f"Mapped exercises: {len(exercise_lookup):,}")
print(f"Ambiguous cross-module exercises excluded: {len(ambiguous_exercises):,}")

Config modules: 27
Mapped exercises: 18,579
Ambiguous cross-module exercises excluded: 0


## Register the parquet and check module mapping coverage

In [3]:
con = duckdb.connect()
con.register("module_lookup", module_lookup)
con.register("exercise_lookup", exercise_lookup)
parquet_sql_path = PARQUET_PATH.as_posix().replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW attempts_enriched AS
    SELECT
        raw.user_id,
        raw.exercise_id,
        raw.classroom_id,
        raw.created_at,
        TRY_CAST(raw.created_at AS TIMESTAMPTZ) AS created_at_ts,
        raw.source_row_number,
        LOWER(TRIM(raw.work_mode)) AS work_mode,
        COALESCE(direct.module_id, exercise.module_id) AS module_id,
        COALESCE(direct.module_code, exercise.module_code) AS module_code,
        COALESCE(direct.module_title, exercise.module_title) AS module_title,
        CASE
            WHEN direct.module_id IS NOT NULL THEN 'direct module id'
            WHEN exercise.module_id IS NOT NULL THEN 'exercise mapping'
            ELSE 'unmapped'
        END AS module_mapping_source
    FROM (
        SELECT *, ROW_NUMBER() OVER () AS source_row_number
        FROM read_parquet('{parquet_sql_path}')
    ) AS raw
    LEFT JOIN module_lookup AS direct
        ON raw.playlist_or_module_id = direct.module_id
    LEFT JOIN exercise_lookup AS exercise
        ON raw.exercise_id = exercise.exercise_id
    """
)

mapping_quality = con.sql(
    """
    SELECT
        work_mode,
        module_mapping_source,
        COUNT(*) AS attempts,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY work_mode), 2)
            AS pct_within_work_mode
    FROM attempts_enriched
    GROUP BY work_mode, module_mapping_source
    ORDER BY work_mode, attempts DESC
    """
).df()
display(mapping_quality)

,work_mode,module_mapping_source,attempts,pct_within_work_mode
0,adaptive-test,direct module id,803851,100.00
1,duo,direct module id,2395,100.00
2,playlist,exercise mapping,1633589,99.63
3,playlist,unmapped,6111,0.37
4,revision,direct module id,1676,100.00
5,undefined,direct module id,14,100.00
6,zpdes,direct module id,3957181,100.00


## Overall counts

In [4]:
overall_summary = con.sql(
    """
    SELECT
        COUNT(DISTINCT user_id) AS unique_students,
        COUNT(DISTINCT exercise_id) AS unique_exercises,
        COUNT(DISTINCT classroom_id) AS unique_classrooms,
        COUNT(*) AS attempts
    FROM attempts_enriched
    """
).df()
display(overall_summary.style.format("{:,}"))

,unique_students,unique_exercises,unique_classrooms,attempts
0,"43,381","18,619","3,545","6,404,817"


## ZPDES versus playlist

Both the target-mode denominator and the all-data denominator are shown explicitly.

In [5]:
mode_list_sql = ", ".join(f"'{mode}'" for mode in TARGET_MODES)
work_mode_summary = con.sql(
    f"""
    WITH mode_counts AS (
        SELECT
            work_mode,
            COUNT(*) AS attempts,
            COUNT(DISTINCT user_id) AS unique_students
        FROM attempts_enriched
        WHERE work_mode IN ({mode_list_sql})
        GROUP BY work_mode
    ),
    target_totals AS (
        SELECT COUNT(*) AS attempts, COUNT(DISTINCT user_id) AS students
        FROM attempts_enriched
        WHERE work_mode IN ({mode_list_sql})
    ),
    all_totals AS (
        SELECT COUNT(*) AS attempts, COUNT(DISTINCT user_id) AS students
        FROM attempts_enriched
    )
    SELECT
        mode_counts.work_mode,
        mode_counts.attempts,
        mode_counts.unique_students,
        ROUND(100.0 * mode_counts.attempts / target_totals.attempts, 2)
            AS pct_target_mode_attempts,
        ROUND(100.0 * mode_counts.unique_students / target_totals.students, 2)
            AS pct_target_mode_students,
        ROUND(100.0 * mode_counts.attempts / all_totals.attempts, 2)
            AS pct_all_attempts,
        ROUND(100.0 * mode_counts.unique_students / all_totals.students, 2)
            AS pct_all_students
    FROM mode_counts
    CROSS JOIN target_totals
    CROSS JOIN all_totals
    ORDER BY mode_counts.attempts DESC
    """
).df()
display(work_mode_summary)

,work_mode,attempts,unique_students,pct_target_mode_attempts,pct_target_mode_students,pct_all_attempts,pct_all_students
0,zpdes,3957181,28489,70.7,75.06,61.78,65.67
1,playlist,1639700,15859,29.3,41.78,25.60,36.56


In [6]:
student_participation = con.sql(
    """
    WITH student_modes AS (
        SELECT
            user_id,
            BOOL_OR(work_mode = 'zpdes') AS used_zpdes,
            BOOL_OR(work_mode = 'playlist') AS used_playlist
        FROM attempts_enriched
        WHERE work_mode IN ('zpdes', 'playlist')
        GROUP BY user_id
    ), participation AS (
        SELECT
            CASE
                WHEN used_zpdes AND used_playlist THEN 'both'
                WHEN used_zpdes THEN 'zpdes only'
                ELSE 'playlist only'
            END AS participation_group
        FROM student_modes
    )
    SELECT
        participation_group,
        COUNT(*) AS unique_students,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_target_mode_students
    FROM participation
    GROUP BY participation_group
    ORDER BY unique_students DESC
    """
).df()
display(student_participation)

,participation_group,unique_students,pct_target_mode_students
0,zpdes only,22098,58.22
1,playlist only,9468,24.94
2,both,6391,16.84


## Contiguous work-mode sequences

A sequence is one student's uninterrupted run of attempts in the same work mode. Attempts are ordered by `created_at`; any change to another work mode, including modes outside ZPDES and playlist, ends the current sequence. Sequence length is the number of attempt rows in the run. Original parquet row order provides deterministic tie-breaking when timestamps are equal.

In [7]:
sequence_order_quality = con.sql(
    """
    WITH timestamp_groups AS (
        SELECT user_id, created_at_ts, COUNT(DISTINCT work_mode) AS work_modes
        FROM attempts_enriched
        WHERE created_at_ts IS NOT NULL
        GROUP BY user_id, created_at_ts
    )
    SELECT
        (SELECT COUNT(*) FROM attempts_enriched WHERE created_at_ts IS NULL)
            AS attempts_without_parsed_timestamp,
        COUNT(*) FILTER (WHERE work_modes > 1) AS student_timestamps_with_multiple_modes
    FROM timestamp_groups
    """
).df()

sequence_summary = con.sql(
    f"""
    WITH ordered AS (
        SELECT
            user_id, exercise_id, work_mode, created_at, created_at_ts, source_row_number,
            LAG(work_mode) OVER (
                PARTITION BY user_id
                ORDER BY created_at_ts NULLS LAST, created_at NULLS LAST, source_row_number
            ) AS previous_work_mode
        FROM attempts_enriched
    ), boundaries AS (
        SELECT
            *,
            CASE
                WHEN previous_work_mode IS NULL OR work_mode <> previous_work_mode THEN 1
                ELSE 0
            END AS starts_sequence
        FROM ordered
    ), numbered AS (
        SELECT
            *,
            SUM(starts_sequence) OVER (
                PARTITION BY user_id
                ORDER BY created_at_ts NULLS LAST, created_at NULLS LAST, source_row_number
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS sequence_number
        FROM boundaries
    ), sequence_lengths AS (
        SELECT
            user_id, sequence_number, work_mode,
            COUNT(*) AS sequence_length
        FROM numbered
        GROUP BY user_id, sequence_number, work_mode
    )
    SELECT
        work_mode,
        COUNT(*) AS unique_sequences,
        COUNT(DISTINCT user_id) AS unique_students,
        SUM(sequence_length) AS attempts_in_sequences,
        ROUND(AVG(sequence_length), 2) AS mean_sequence_length,
        MEDIAN(sequence_length) AS median_sequence_length
    FROM sequence_lengths
    WHERE work_mode IN ({mode_list_sql})
    GROUP BY work_mode
    ORDER BY unique_sequences DESC
    """
).df()

display(sequence_order_quality)
display(sequence_summary)

,attempts_without_parsed_timestamp,student_timestamps_with_multiple_modes
0,0,0


,work_mode,unique_sequences,unique_students,attempts_in_sequences,mean_sequence_length,median_sequence_length
0,zpdes,49876,28489,3957181.0,79.34,45.0
1,playlist,18455,15859,1639700.0,88.85,43.0


## Counts by module

Module codes distinguish modules whose displayed long number is reused across French and mathematics (for example, `M1` and `M101`).

In [8]:
module_overview = con.sql(
    f"""
    SELECT
        COALESCE(module_code, 'Unmapped') AS module_code,
        COALESCE(module_title, 'Unmapped') AS module_title,
        COUNT(DISTINCT user_id) AS unique_students,
        COUNT(DISTINCT exercise_id) AS unique_exercises,
        COUNT(DISTINCT classroom_id) AS unique_classrooms,
        COUNT(*) AS attempts,
        COUNT(*) FILTER (WHERE work_mode IN ({mode_list_sql})) AS target_mode_attempts,
        COUNT(DISTINCT user_id) FILTER (WHERE work_mode IN ({mode_list_sql}))
            AS target_mode_students,
        COALESCE(TRY_CAST(REGEXP_EXTRACT(module_code, '[0-9]+') AS INTEGER), 999999)
            AS module_sort
    FROM attempts_enriched
    GROUP BY module_code, module_title
    ORDER BY module_sort, module_code
    """
).df()
module_overview = module_overview.drop(columns="module_sort")
display(module_overview)

,module_code,module_title,unique_students,unique_exercises,unique_classrooms,attempts,target_mode_attempts,target_mode_students
0,M1,Réapprentissage des correspondances graphèmes-...,4487,2117,1320,272927,241907,2431
1,M2,Fluence de décodage de la lecture,2213,1310,731,69880,55307,1071
2,M3,De l'oral à l'écrit,1177,137,365,25208,25175,1173
3,M4,Améliorer la compréhension des textes,4764,630,811,419595,358778,3484
4,M5,Amélioration de la production écrite et de l'a...,2176,156,443,36613,36613,2176
5,M6,Syntaxe niveau 1,6539,697,833,611612,546302,5275
6,M7,Syntaxe niveau 2,1660,1327,313,132038,119349,1281
7,M8,Orthographe niveau 1,5936,509,808,465632,393653,4591
8,M9,Orthographe niveau 2,1789,671,377,135699,122186,1366
9,M10,Lexique niveau 1,2321,362,468,163283,144446,1644


## ZPDES versus playlist by module

`pct_module_target_*` compares ZPDES with playlist inside each module. `pct_module_all_*` uses every work mode in that module as the denominator. Student percentages can overlap because a student can use both modes within a module.

In [9]:
module_work_mode_summary = con.sql(
    f"""
    WITH labeled AS (
        SELECT
            COALESCE(module_code, 'Unmapped') AS module_code,
            COALESCE(module_title, 'Unmapped') AS module_title,
            work_mode, user_id
        FROM attempts_enriched
    ), mode_counts AS (
        SELECT
            module_code, module_title, work_mode,
            COUNT(*) AS attempts,
            COUNT(DISTINCT user_id) AS unique_students
        FROM labeled
        WHERE work_mode IN ({mode_list_sql})
        GROUP BY module_code, module_title, work_mode
    ), module_target_totals AS (
        SELECT
            module_code, module_title,
            COUNT(*) AS attempts,
            COUNT(DISTINCT user_id) AS students
        FROM labeled
        WHERE work_mode IN ({mode_list_sql})
        GROUP BY module_code, module_title
    ), module_all_totals AS (
        SELECT
            module_code, module_title,
            COUNT(*) AS attempts,
            COUNT(DISTINCT user_id) AS students
        FROM labeled
        GROUP BY module_code, module_title
    )
    SELECT
        mode_counts.module_code,
        mode_counts.module_title,
        mode_counts.work_mode,
        mode_counts.attempts,
        mode_counts.unique_students,
        ROUND(100.0 * mode_counts.attempts / module_target_totals.attempts, 2)
            AS pct_module_target_attempts,
        ROUND(100.0 * mode_counts.unique_students / module_target_totals.students, 2)
            AS pct_module_target_students,
        ROUND(100.0 * mode_counts.attempts / module_all_totals.attempts, 2)
            AS pct_module_all_attempts,
        ROUND(100.0 * mode_counts.unique_students / module_all_totals.students, 2)
            AS pct_module_all_students,
        COALESCE(TRY_CAST(REGEXP_EXTRACT(mode_counts.module_code, '[0-9]+') AS INTEGER), 999999)
            AS module_sort
    FROM mode_counts
    JOIN module_target_totals USING (module_code, module_title)
    JOIN module_all_totals USING (module_code, module_title)
    ORDER BY module_sort, mode_counts.module_code, mode_counts.work_mode
    """
).df()
module_work_mode_summary = module_work_mode_summary.drop(columns="module_sort")
display(module_work_mode_summary)

,module_code,module_title,work_mode,attempts,unique_students,pct_module_target_attempts,pct_module_target_students,pct_module_all_attempts,pct_module_all_students
0,M1,Réapprentissage des correspondances graphèmes-...,playlist,20190,295,8.35,12.13,7.40,6.57
1,M1,Réapprentissage des correspondances graphèmes-...,zpdes,221717,2161,91.65,88.89,81.24,48.16
2,M2,Fluence de décodage de la lecture,playlist,22252,259,40.23,24.18,31.84,11.70
3,M2,Fluence de décodage de la lecture,zpdes,33055,824,59.77,76.94,47.30,37.23
4,M3,De l'oral à l'écrit,playlist,11027,304,43.80,25.92,43.74,25.83
5,M3,De l'oral à l'écrit,zpdes,14148,884,56.20,75.36,56.13,75.11
6,M4,Améliorer la compréhension des textes,playlist,94974,894,26.47,25.66,22.63,18.77
7,M4,Améliorer la compréhension des textes,zpdes,263804,2647,73.53,75.98,62.87,55.56
8,M5,Amélioration de la production écrite et de l'a...,playlist,14957,730,40.85,33.55,40.85,33.55
9,M5,Amélioration de la production écrite et de l'a...,zpdes,21656,1503,59.15,69.07,59.15,69.07


## Optional CSV export

Set `EXPORT_RESULTS = True` in the setup cell to refresh machine-readable outputs.

In [10]:
if EXPORT_RESULTS:
    output_dir = PROJECT_ROOT / "notebooks" / "outputs" / "mia_interaction_overview"
    output_dir.mkdir(parents=True, exist_ok=True)
    results = {
        "overall_summary": overall_summary,
        "mapping_quality": mapping_quality,
        "work_mode_summary": work_mode_summary,
        "student_participation": student_participation,
        "sequence_order_quality": sequence_order_quality,
        "sequence_summary": sequence_summary,
        "module_overview": module_overview,
        "module_work_mode_summary": module_work_mode_summary,
    }
    for name, frame in results.items():
        frame.to_csv(output_dir / f"{name}.csv", index=False)
    print(f"Exported {len(results)} tables to {output_dir}")
else:
    print("CSV export disabled. Set EXPORT_RESULTS = True to enable it.")

CSV export disabled. Set EXPORT_RESULTS = True to enable it.
